<div style="font-family: Inter, Segoe UI, Arial, sans-serif; padding: 28px 30px; border: 1px solid #ccd6e0; border-radius: 8px; background: #101820; color: #f8fafc;">
  <div style="font-size: 12px; font-weight: 800; text-transform: uppercase; color: #f2c14e; margin-bottom: 8px;">Local AI canvas on Colab</div>
  <div style="font-family: Georgia, Times New Roman, serif; font-size: 46px; line-height: 1; font-weight: 700; color: #fff7df; text-shadow: 0 10px 30px rgba(0, 0, 0, 0.35);">Expandiffusion</div>
  <div style="width: 88px; height: 3px; background: #f2c14e; margin: 18px 0 16px;"></div>
  <p style="font-size: 16px; line-height: 1.55; max-width: 760px; margin: 0; color: #dbe5ee;">Run the notebook once, open the temporary Studio link, and start editing from the browser.</p>
</div>

### Start

Use **Runtime > Run all**. The notebook prepares the runtime, starts the API and web UI, then shows the temporary Studio link.

The setup is split into short folded steps and does not ask for interactive input.


In [ ]:
#@title Prepare runtime
#@markdown Defines paths, ports and small helpers used by the next steps.
import json
import os
import subprocess
import sys
import time
import urllib.request
from html import escape
from pathlib import Path

from IPython.display import HTML, display
from google.colab import output

REPO_URL = "https://github.com/PailletJuanPablo/expandiffusion.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/expandiffusion")
API_PORT = 8011
WEB_PORT = 5180


def run(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)


def wait_for_json(url, label, timeout_seconds=180):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                return json.loads(response.read().decode("utf-8"))
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{label} did not become ready at {url}: {last_error}")


def wait_for_text(url, label, timeout_seconds=180):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                return response.status, response.read(256).decode("utf-8", errors="replace")
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{label} did not become ready at {url}: {last_error}")

print("Checking runtime hardware...")
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
#@title Get the latest project code
#@markdown Clones the repo, or refreshes the existing Colab checkout if this runtime was already used.
if (PROJECT_DIR / ".git").exists():
    print(f"Refreshing existing checkout at {PROJECT_DIR}")
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=PROJECT_DIR)
elif (PROJECT_DIR / "apps" / "api").exists():
    print(f"Using existing checkout at {PROJECT_DIR}")
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(PROJECT_DIR)])

api_dir = PROJECT_DIR / "apps" / "api"
web_dir = PROJECT_DIR / "apps" / "web"


In [ ]:
#@title Install dependencies
#@markdown Installs the API and web dependencies required by this runtime.
run([sys.executable, "-m", "pip", "install", "-e", ".[dev,diffusers]"], cwd=api_dir)
run(["npm", "--prefix", str(web_dir), "ci"])


In [ ]:
#@title Launch Studio
#@markdown Starts the API and web UI, waits until they are ready, then shows the temporary Studio link.
env_values = {
    "EXPANDIFFUSION_PLUGIN_DIR": "apps/api/plugins",
    "EXPANDIFFUSION_API_HOST": "0.0.0.0",
    "EXPANDIFFUSION_API_PORT": str(API_PORT),
    "EXPANDIFFUSION_WEB_HOST": "0.0.0.0",
    "EXPANDIFFUSION_WEB_PORT": str(WEB_PORT),
    "EXPANDIFFUSION_WEB_ALLOWED_HOSTS": "*",
    "EXPANDIFFUSION_PYTHON": sys.executable,
}
for key, value in env_values.items():
    os.environ[key] = value

(PROJECT_DIR / ".env").write_text(
    "\n".join(f"{key}={value}" for key, value in env_values.items()) + "\n",
    encoding="utf-8",
)

if "EXPANDIFFUSION_PROCESS" in globals() and EXPANDIFFUSION_PROCESS.poll() is None:
    EXPANDIFFUSION_PROCESS.terminate()
    EXPANDIFFUSION_PROCESS.wait(timeout=20)

log_dir = PROJECT_DIR / "colab-logs"
log_dir.mkdir(exist_ok=True)
log_file = open(log_dir / "dev.log", "w", encoding="utf-8")
EXPANDIFFUSION_PROCESS = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

api_health = wait_for_json(f"http://127.0.0.1:{API_PORT}/api/health", "Expandiffusion API")
if not api_health.get("ok"):
    raise RuntimeError(f"API health check failed: {api_health}")

web_status, _web_preview = wait_for_text(f"http://127.0.0.1:{WEB_PORT}/", "Expandiffusion web UI")
api_url = output.eval_js(f"google.colab.kernel.proxyPort({API_PORT})")
ui_url = output.eval_js(f"google.colab.kernel.proxyPort({WEB_PORT})")
if not isinstance(api_url, str) or not api_url.startswith("http"):
    raise RuntimeError(f"Colab did not return a valid API proxy URL: {api_url!r}")
if not isinstance(ui_url, str) or not ui_url.startswith("http"):
    raise RuntimeError(f"Colab did not return a valid UI proxy URL: {ui_url!r}")

escaped_ui_url = escape(ui_url, quote=True)
print("API ready:", bool(api_health.get("ok")))
print("Studio ready:", web_status == 200)
display(HTML(f"""
<div style="font-family: Inter, Segoe UI, Arial, sans-serif; padding: 18px 20px; border: 1px solid #d7dee8; border-radius: 8px; background: #f8fafc; color: #0f172a;">
  <div style="font-size: 12px; font-weight: 800; text-transform: uppercase; color: #0f766e; margin-bottom: 6px;">Studio ready</div>
  <div style="font-size: 22px; font-weight: 800; margin-bottom: 8px;">Expandiffusion is running</div>
  <p style="font-size: 14px; line-height: 1.5; color: #475569; margin: 0 0 14px;">This Colab link is temporary and stays active while the runtime is running.</p>
  <a href="{escaped_ui_url}" target="_blank" rel="noopener noreferrer" style="display: inline-block; padding: 10px 14px; border-radius: 8px; background: #0f766e; color: #ffffff; font-weight: 800; text-decoration: none;">Open Studio</a>
  <div style="font-size: 12px; color: #64748b; margin-top: 12px; word-break: break-all;">{escaped_ui_url}</div>
</div>
"""))
